In [ ]:
# top 300 QTSA test with RAG integration

import torch, os, re
import json
import time
import textwrap
import numpy as np
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
BASE_DIR = "path to your project directory"  # Change this to your project directory
MODEL_DIR = "path to your model directory"  # Change this to your model directory
INPUT_FILE = "path to your input file"  # Change this to your input file
OUTPUT_FILE = "path to your output file"  # Change this to your output file
CONTEXT_FILE = "path to your context output file"  # Change this to your context output file

# ============== RAG Functions ==============
def load_rag_components():
    # load vertor database
    vectordb = Chroma(collection_name="database name", persist_directory="your directory path to vector database")
    print("Collection count:", vectordb._collection.count())

    data = vectordb.get()
    raw_text = data["documents"]
    raw_metadata = data["metadatas"]
    
    docs = []
    for i, (text, meta) in enumerate(zip(raw_text, raw_metadata)):
        m = dict(meta) if meta is not None else {}
        m["chunk_idx"] = i
        docs.append(Document(page_content=text, metadata=m))

    bge_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    # reranker = FlagReranker("BAAI/bge-reranker-base", use_fp16=True)
    # bm25 = BM25Retriever.from_documents(docs)
    # bm25.k = 15
    
    return {
        'vectordb': vectordb,
        'raw_text': raw_text,
        'raw_metadata': raw_metadata,
        'bge_model': bge_model,
        'docs': docs
    }

def get_rag_context(question, rag_components, top_k_dense=3):
    """get context according to each question"""

    vectordb = rag_components['vectordb']
    raw_text = rag_components['raw_text']
    raw_metadata = rag_components['raw_metadata']
    bge_model = rag_components['bge_model']
    # reranker = rag_components['reranker']
    # bm25 = rag_components['bm25']
    
    # ====== Method1: vector similarity comparison ======
    q = bge_model.encode([question], return_dense=True, return_sparse=False)
    qvec = np.asarray(q["dense_vecs"][0], dtype=np.float32)
    norm = np.linalg.norm(qvec)
    if norm != 0:
        qvec = qvec / norm

    results = vectordb._collection.query(
        query_embeddings=[qvec],
        n_results=top_k_dense,
        include=["documents", "metadatas", "distances"]
    )
    
    # # ====== Method2: key words comparison ======
    # bm25_docs = bm25_docs = bm25.invoke(question) if hasattr(bm25, 'invoke') else bm25(question)
    
    # # ====== Reciprocal Rank Fusion ======
    # dense_pairs = []
    # for doc, meta, dist, _id in zip(results["documents"][0], results["metadatas"][0], 
    #                                results["distances"][0], results["ids"][0]):
    #     score = 1.0 - float(dist)
    #     idx = meta.get("chunk_idx", -1)
    #     if idx >= 0:
    #         dense_pairs.append((idx, score))

    # sparse_pairs = []
    # for rank, d in enumerate(bm25_docs, start=1):
    #     idx = d.metadata["chunk_idx"]
    #     sparse_pairs.append((idx, 1.0 / rank))

    # dense_ordered = [i for i, _ in sorted(dense_pairs, key=lambda x: x[1], reverse=True)]
    # sparse_ordered = [i for i, _ in sorted(sparse_pairs, key=lambda x: x[1], reverse=True)]

    # rrf_k = 60
    # rrf_scores = defaultdict(float)
    
    # for lst in [dense_ordered, sparse_ordered]:
    #     for rank, i in enumerate(lst, start=1):
    #         rrf_scores[i] += 1.0 / (rrf_k + rank)

    # origin = {}
    # for i, _ in dense_pairs:
    #     origin.setdefault(i, "dense")
    # for i, _ in sparse_pairs:
    #     origin.setdefault(i, "sparse")

    # top_k_rrf = 20
    # fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_rrf]
    # fused_idx = [i for i, _ in fused]
    
    # # ====== reranking======
    # pairs = [[question, raw_text[i]] for i in fused_idx]
    # ce_scores = reranker.compute_score(pairs, batch_size=16)
    # ranked = sorted(zip(fused_idx, ce_scores), key=lambda x: x[1], reverse=True)
    
    # final_indices_scores = ranked[:top_k_final]

    # ====== get context ======
    # create citations for llm
    context_lines = []
    detailed_sources = []
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
        src = meta.get("source") or meta.get("file_path") or ""
        page = meta.get("page")
        tag = f"{src}" + (f":p{page}" if page is not None else "")
        snippet = textwrap.shorten(doc.strip().replace("\n", " "), width=1200, placeholder=" ...")
        context_lines.append(f"[S{i}] {tag} :: {snippet}")
        detailed_sources.append({
        "citation_id": i,
        "source": src,
        "page": page,
        "text_snippet": doc.strip(),
        "text_length": len(doc.strip())
    })   
    packed_context = "\n".join(context_lines)
    
    return packed_context, detailed_sources

def save_context_to_jsonl(context_data, context_file_path):
    with open(context_file_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(context_data, ensure_ascii=False) + "\n")
        
# ===========================main()========================
def main():
    base_model_path = MODEL_DIR
    tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        device_map="auto",
        torch_dtype=torch.float16
    )
    model.eval()

    print("Loading RAG components...")
    rag_components = load_rag_components()
    print("RAG components loaded successfully!")
    
    # file path
    jsonl_path = INPUT_FILE
    output_path = OUTPUT_FILE
    context_output_path = CONTEXT_FILE
    
    total_start_time = time.time()
    
    results = []
    max_items = 300
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
                
            if i >= max_items:
                break
            
            try:
                data = json.loads(line.strip())
                question = data.get("question", "")
                correct_answer = data.get("answer", "")
                id_num = data.get("id", i + 1)
                
                print(f"\n[{i+1}/{max_items}] Processing ID {id_num}: {question[:60]}...")
                
                # ====== Retrieving RAG context ======
                print("  Retrieving RAG context...")
                rag_start = time.time()
                context, detailed_sources = get_rag_context(question, rag_components)
                rag_time = time.time() - rag_start

                # save RAG context into file
                context_data = {
                    "id": id_num,
                    "question": question,
                    "rag_time": rag_time,
                    "retrieved_sources_count": len(detailed_sources),
                    "detailed_sources": detailed_sources,
                    "concatenated_context": context,
                }
                save_context_to_jsonl(context_data, context_output_path)

                # ====== set up prompt ======
                system_prompt = "You are a Radio Frequency expert. Please answer questions using concise and professional language, providing the core content directly without excessive explanation."

                user_prompt = f"""Based on the following reference information, please answer the question:
                
                Reference Information:
                {context}
                
                Question: {question}
                
                Please provide a concise and accurate answer based on the reference information above. If the information is not sufficient, use your professional knowledge to answer."""
                
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
                
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False  # ban thinking mode
                )
                
                start_time = time.time()
                
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
                input_tokens_count = inputs.input_ids.shape[1]
                print(f"  Input tokens: {input_tokens_count}")
                
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=16000,  
                        repetition_penalty=1.2,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                
                output_tokens_count = outputs[0].shape[0] - inputs.input_ids.shape[1]
                full_response = tokenizer.decode(
                    outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
                )
                
                inference_time = time.time() - start_time
                print(f"  Output tokens: {output_tokens_count}")
                print(f"  Inference time: {inference_time:.2f}s")
                
                result = {
                    "id": id_num,
                    "question": question,
                    "rag_context": context,
                    "model_output": full_response,
                    "correct_answer": correct_answer,
                    "rag_time": rag_time,
                    "inference_time": inference_time,
                    "total_time": rag_time + inference_time,
                    "input_tokens": input_tokens_count,
                    "output_tokens": output_tokens_count
                }
                results.append(result)

                print(f"  Model response: {full_response[:80]}...")
                
            except json.JSONDecodeError as e:
                print(f"Line {i+1} JSON decode error: {e}")
            except Exception as e:
                print(f"Error processing line {i+1}: {e}")
                import traceback
                traceback.print_exc()

    total_end_time = time.time()
    total_processing_time = total_end_time - total_start_time
    
    if results:
        avg_time_per_question = total_processing_time / len(results)

        avg_rag_time = sum(r["rag_time"] for r in results) / len(results)
        avg_inference_time = sum(r["inference_time"] for r in results) / len(results)
        avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
        avg_output_tokens = sum(r["output_tokens"] for r in results) / len(results)
        
        print(f"\n{'='*60}")
        print(f"SUMMARY:")
        print(f"{'='*60}")
        print(f"Total questions processed: {len(results)}")
        print(f"Total processing time: {total_processing_time:.2f}s")
        print(f"Average time per question: {avg_time_per_question:.2f}s")
        print(f"  - Average RAG time: {avg_rag_time:.2f}s")
        print(f"  - Average inference time: {avg_inference_time:.2f}s")
        print(f"Average input tokens: {avg_input_tokens:.1f}")
        print(f"Average output tokens: {avg_output_tokens:.1f}")
        print(f"{'='*60}")

    with open(output_path, "w", encoding="utf-8") as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
    
    print(f"\nComplete! Results saved to: {output_path}")
    print(f"RAG contexts saved to: {context_output_path}")
    print(f"Processed {len(results)} questions")

if __name__ == "__main__":
    main()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading RAG components...


/tmp/ipykernel_3079610/2139741050.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(collection_name="rag_db_remove_textbook_duplicates", persist_directory="/home/jw161/chatbot/chromadb")


Collection count: 33106


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

RAG components loaded successfully!

[1/300] Processing ID 4973: In the design of a cross-coupled LC oscillator, the loop gai...
  Retrieving RAG context...


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Input tokens: 1493
  Output tokens: 113
  Inference time: 3.98s
  Model response: The correct answer is **D**.

Explanation:  
Since the loop gain $ T(f_o) = (g_m...

[2/300] Processing ID 2720: In using a nonlinear device such as a diode or BJT mixer, wh...
  Retrieving RAG context...
  Input tokens: 1101
  Output tokens: 154
  Inference time: 5.10s
  Model response: The correct answer is **A**. 

**Explanation**:  
According to the reference inf...

[3/300] Processing ID 841: In the design of a Lange coupler modeled as a four-wire coup...
  Retrieving RAG context...
  Input tokens: 1662
  Output tokens: 117
  Inference time: 3.87s
  Model response: The correct expression is:

**C. $Z_{e4} = \frac{Z_{0o} + Z_{0e}}{3Z_{0o} + Z_{0...

[4/300] Processing ID 4441: In the design of a maximally flat Class F2 amplifier employi...
  Retrieving RAG context...
  Input tokens: 1985
  Output tokens: 103
  Inference time: 3.44s
  Model response: The correct ratio is $ \frac{V_{m2}}{V_m} = \frac{